# Data Quality Audit

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

prices_22 = pd.read_csv('data/raw/precios-eess-2022.csv')
prices_23 = pd.read_csv('data/raw/precios-eess-2023.csv')
prices_24 = pd.read_csv('data/raw/precios-eess-2024.csv')
prices_25 = pd.read_csv('data/raw/precios-eess-2025-.csv')

In [67]:
prices_dict = {
    '2022': prices_22,
    '2023': prices_23,
    '2024': prices_24,
    '2025': prices_25
}

### Column names

In [68]:
for year, df in prices_dict.items():
    print(year)
    print(list(df.columns))

2022
['anio', 'mes', 'operador', 'nro_inscripcion', 'bandera', 'fecha_de_baja', 'cuit', 'tipo_negocio', 'direccion', 'localidad', 'provincia', 'producto', 'canal_de_comercializacion', 'precio_sin_impuestos', 'precio_con_impuestos', 'volumen', 'precio_surtidor', 'no_novimientos', 'exentos']
2023
['anio', 'mes', 'operador', 'nro_inscripcion', 'bandera', 'fecha_de_baja', 'cuit', 'tipo_negocio', 'direccion', 'localidad', 'provincia', 'producto', 'canal_de_comercializacion', 'precio_sin_impuestos', 'precio_con_impuestos', 'volumen', 'precio_surtidor', 'no_novimientos', 'exentos']
2024
['anio', 'mes', 'operador', 'nro_inscripcion', 'bandera', 'fecha_de_baja', 'cuit', 'tipo_negocio', 'direccion', 'localidad', 'provincia', 'producto', 'canal_de_comercializacion', 'precio_sin_impuestos', 'precio_con_impuestos', 'volumen', 'precio_surtidor', 'no_novimientos', 'exentos']
2025
['fecha', 'periodo', 'operador', 'nro_inscripcion', 'bandera', 'fecha_de_baja', 'cuit', 'tipo_negocio', 'direccion', 'loca

2025 dataset introduces date and period instead of year and month plus seven new fields that describe different taxes, and -apparently- change "no_novimientos" and "exentos" to "no_movimientos" and "excentos".

### 2025 period column inspection

In [69]:
prices_25['periodo'].unique()

array(['2024/12', '2025/01', '2025/02', '2025/03', '2025/04', '2025/05',
       '2025/06', '2025/07', '2025/08', '2025/09', '2025/10', '2025/11',
       '2025/12', '2026/01', '2026/02', '2026/03', '2026/04', '2026/05',
       '2026/06', '2026/07'], dtype=object)

"2025" dataset covers from december-24 up to july-26. Since the scope of the project is bounded to the four most recent completed years (2022-2025), 2026 records have to be excluded. Besides that, a futher inspection -and comparison- of december 2024 is performed later in this notebook.

#### Year and month columns extraction in prices-25

In [70]:
prices_25['anio'] = prices_25['periodo'].str.split("/", n=1).str[0]
prices_25['mes'] = prices_25['periodo'].str.split("/", n=1).str[1]

### Year & month: coverage and missingness

In [71]:
for year, df in prices_dict.items():
    print(f"{year}:")
    print(f"Shape: {df.shape}")
    print(f"Year unique values: {df['anio'].unique()}")
    print(f"Month unique values: {df['mes'].unique()}")
    print(f"Null year values: {df['anio'].isna().sum()}")
    print("==="*15)

2022:
Shape: (261473, 19)
Year unique values: [2022]
Month unique values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Null year values: 0
2023:
Shape: (259806, 19)
Year unique values: [2023]
Month unique values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Null year values: 0
2024:
Shape: (256798, 19)
Year unique values: [2024]
Month unique values: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Null year values: 0
2025:
Shape: (431173, 28)
Year unique values: ['2024' '2025' '2026']
Month unique values: ['12' '01' '02' '03' '04' '05' '06' '07' '08' '09' '10' '11']
Null year values: 0


The first three datasets don't have year overlapping, and every row in all four datasets have a correct non-null year value.

Besides the fact of the last datasets month and year values being strings, there is no other inconsistency regarding to the compatibility between datasets.

### Data types

In [72]:
dtypes_22_24 = pd.concat([df.dtypes.rename(year) for year, df in prices_dict.items()], axis=1, join="outer")
dtypes_22_24

,2022,2023,2024,2025
anio,int64,int64,int64,object
mes,int64,int64,int64,object
operador,object,object,object,object
nro_inscripcion,int64,int64,int64,int64
bandera,object,object,object,object
fecha_de_baja,object,object,object,object
cuit,object,object,object,object
tipo_negocio,object,object,object,object
direccion,object,object,object,object
localidad,object,object,object,object


fecha_de_baja and fecha (from prices-25) are expected to be date columns, but registered as object. And, as seen above, prices-25 anio and mes remain as object, while expected as integers.

#### Prices-25 year & month: dtype correction

In [73]:
prices_25['anio'] = prices_25['anio'].astype('int64')
prices_25['mes'] = prices_25['mes'].astype('int64')

### Canonical naming of `no_movimientos` and `exentos`.

In [74]:
for year, df in prices_dict.items():
    if year != "2025":
        print(year)
        print(f"no_novimientos values: {df['no_novimientos'].unique()}")
        print(f"exentos values: {df['exentos'].unique()}\n" + "==="*12)
    else:
        print(year)
        print(f"no_movimientos values: {df['no_movimientos'].unique()}")
        print(f"excentos values: {df['excentos'].unique()}\n" + "==="*12)

2022
no_novimientos values: ['NO' 'SI']
exentos values: ['f' 't' nan]
2023
no_novimientos values: ['NO' 'SI']
exentos values: ['f' nan 't']
2024
no_novimientos values: ['NO' 'SI']
exentos values: ['f' nan 't']
2025
no_movimientos values: ['NO' 'SI']
excentos values: ['f' nan 't']


We can confirm that these columns correspond to the same information.

Therefore, we can rename this fields to an only canonical name.

#### Column renaming
* Prices 22-24: `no_novimientos` -> `no_movimientos`
* Prices 25: `excentos` -> `exentos`

In [75]:
for year, df in prices_dict.items():
    if year != "2025":
        df.rename(columns={"no_novimientos": "no_movimientos"}, inplace=True)
    else:
        df.rename(columns={"excentos": "exentos"}, inplace=True)

### Duplicated rows

#### Full row

In [76]:
for year, df in prices_dict.items():
    print(year)
    print(f"Duplicated count: {df.duplicated().sum()}\n" + "==="*12)

2022
Duplicated count: 3
2023
Duplicated count: 5
2024
Duplicated count: 6
2025
Duplicated count: 14


#### Candidate key

In [77]:
cand_key = ['anio', 'mes', 'nro_inscripcion', 'canal_de_comercializacion', 'producto', 'exentos']
for year, df in prices_dict.items():
    print(year)
    print(f"Duplicated count: {df.duplicated(subset=cand_key).sum()}\n" + "==="*12)

2022
Duplicated count: 3
2023
Duplicated count: 5
2024
Duplicated count: 6
2025
Duplicated count: 15


### December 2024 inspection

In [78]:
selected_columns = [
    'anio',
    'mes',
    'cuit',
    'bandera',
    'operador',
    'nro_inscripcion',
    'provincia',
    'localidad',
    'direccion',
    'tipo_negocio',
    'canal_de_comercializacion',
    'producto',
    'volumen',
    'precio_sin_impuestos',
    'precio_con_impuestos',
    'precio_surtidor',
    'exentos',
    'fecha_de_baja',
    'no_movimientos'
]
comparison_columns = [col for col in selected_columns if col not in ('fecha_de_baja', '_merge')]

In [79]:
prices_dic24_1 = prices_24[(prices_24['anio']==2024) & (prices_24['mes']==12)][selected_columns]
prices_dic24_2 = prices_25[(prices_25['anio']==2024) & (prices_25['mes']==12)][selected_columns]
print(f"Number of rows\nPrices-24: {prices_dic24_1.shape[0]} \nPrices-25: {prices_dic24_2.shape[0]} \n" + "==="*15)

intersection = pd.merge(prices_dic24_1, prices_dic24_2, how="inner")
print(f"Number of exact duplicates between dataframes: {intersection.shape[0]}\n" + "==="*15)

intersected_keys = pd.merge(prices_dic24_1[cand_key], prices_dic24_2[cand_key], how="inner")
print(f"Number of duplicated keys between dataframes: {intersected_keys.shape[0]}\n" + "==="*15)

intersection_no_baja = pd.merge(prices_dic24_1[comparison_columns], prices_dic24_2[comparison_columns], how="inner")
print(f"Number of matching rows excluding fecha_de_baja and _merge: {intersection_no_baja.shape[0]}\n" + "==="*15)

Number of rows
Prices-24: 20700 
Prices-25: 21601 
Number of exact duplicates between dataframes: 19139
Number of duplicated keys between dataframes: 20686
Number of matching rows excluding fecha_de_baja and _merge: 19316


In [80]:
union = pd.merge(prices_dic24_1, prices_dic24_2, how="outer", indicator=True)
union_duplicated_key = union[union.duplicated(subset=cand_key, keep=False)]

In [81]:
union_dk_no_mov = union_duplicated_key[['_merge', 'no_movimientos']].value_counts()
print("Records with duplicated key between prices-24 and prices-25, with or without movements")
print(f"Prices-24: \n{union_dk_no_mov.loc['left_only']}\n" + "==="*10)
print(f"Prices-25: \n{union_dk_no_mov.loc['right_only']}\n")

Records with duplicated key between prices-24 and prices-25, with or without movements
Prices-24: 
no_movimientos
NO    1535
SI      12
Name: count, dtype: int64
Prices-25: 
no_movimientos
NO    1535
SI      12
Name: count, dtype: int64



In [82]:
union['provenance'] = union['_merge'].map({"left_only": "prices_2024", "right_only": "prices_2025", "both": "both_2024_2025"})

### Record provenance traceability

In [83]:
for year, df in prices_dict.items():
    df['provenance'] = f"prices_{year}"

### Data concatenation

In [84]:
final_columns = selected_columns + ['provenance']

In [85]:
final_dataframes = [
    prices_22[final_columns],
    prices_23[final_columns],
    prices_24[prices_24['mes']!=12][final_columns],
    union[final_columns],
    prices_25[prices_25['periodo']!='2024/12'][final_columns],
]

In [86]:
prices = pd.DataFrame(pd.concat(final_dataframes))
n, m = prices.shape
print(f"Resulting dataframe shape: {n} x {m}\n" + "==="*15)
print(f"Number of duplicated full rows: {prices.duplicated().sum()}\n" + "==="*15)
print(f"Number of rows with duplicated keys: {prices.duplicated(subset=cand_key).sum()}\n" + "==="*15)


Resulting dataframe shape: 1190111 x 20
Number of duplicated full rows: 28
Number of rows with duplicated keys: 1576


#### Exclusion: 2026 records

In [87]:
print(f"2026 records: {len(prices[prices['anio']==2026])}")

prices = prices[prices['anio']!=2026]
print(f"Resulting rows: {prices.shape[0]}")

2026 records: 148321
Resulting rows: 1041790


#### Deduplication: full rows

In [88]:
prices = prices.drop_duplicates()
print(f"Resulting rows: {prices.shape[0]}. Duplicated: {prices.duplicated().sum()}")

Resulting rows: 1041767. Duplicated: 0


### Flag: duplicated candidate key

In [ ]:
prices['duplicated_key'] = prices.duplicated(subset=cand_key, keep=False)

***

In [90]:
prices.describe(percentiles=[0.5])

,anio,mes,nro_inscripcion,volumen,precio_sin_impuestos,precio_con_impuestos,precio_surtidor
count,1.041767e+06,1.041767e+06,1.041767e+06,1.041767e+06,1.041767e+06,1.041767e+06,1.041767e+06
mean,2.023499e+03,6.501519e+00,5.693367e+03,9.467414e+03,5.379281e+02,7.261687e+02,6.283246e+02
std,1.119603e+00,3.455079e+00,3.418284e+03,4.392081e+05,4.176822e+02,5.730653e+02,5.951478e+02
min,2.022000e+03,1.000000e+00,1.010000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.023000e+03,6.000000e+00,6.003000e+03,5.800300e+01,3.966120e+02,4.990000e+02,3.152000e+02
max,2.025000e+03,1.200000e+01,1.152500e+04,3.550410e+08,4.910000e+03,4.999000e+03,5.000000e+03


There's no negativeness in the numeric fields, but there are zeros in prices and volume.

In [91]:
print("Column missing values")
for col in prices.columns:
    print(f"{col:28}: {prices[col].isna().sum()}")

Column missing values
anio                        : 0
mes                         : 0
cuit                        : 0
bandera                     : 0
operador                    : 0
nro_inscripcion             : 0
provincia                   : 0
localidad                   : 0
direccion                   : 14
tipo_negocio                : 0
canal_de_comercializacion   : 0
producto                    : 0
volumen                     : 0
precio_sin_impuestos        : 0
precio_con_impuestos        : 0
precio_surtidor             : 0
exentos                     : 9489
fecha_de_baja               : 1004729
no_movimientos              : 0
provenance                  : 0
duplicated_key              : 0


In [92]:
#for col in prices.select_dtypes(include=['object']).columns:
#    n = prices[col].nunique()
#    if n > 10: 
#        print(f"Column {col} has {n} unique values. \nTop 10 categories:") 
#    else: print (f"Column {col} has {n} unique values. \nCategories:")
#    display(prices[col].value_counts().head(10).reset_index())

In [93]:
import unicodedata
import re

def remove_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(
        char for char in normalized if not unicodedata.combining(char)
    )

def normalize_text(text: str) -> str:
    """Normalize a non-null text value without changing its semantics."""
    if text is None:
        return text
    text = str(text).strip().lower()
    text = remove_accents(text)
    text = re.sub(r"\s+", " ", text)
    return text

In [94]:
prices_norm_txt = prices.copy()
for col in prices.select_dtypes(include=['object']).columns:
    prices_norm_txt[col] = prices[col].apply(normalize_text)

In [95]:
print("Text normalization cardinality test:\n" + " "*15 + "Before | After")
for col in prices.select_dtypes(include=['object']).columns:
    if col != 'fecha_de_baja' and prices[col].nunique() >= 10:
        before = prices[col].nunique()
        after = prices_norm_txt[col].nunique()
        print(f"{col:15}{before:7}|{after:6} diff={before-after}")

Text normalization cardinality test:
               Before | After
cuit              3819|  3819 diff=0
bandera             23|    23 diff=0
operador          4358|  4336 diff=22
provincia           24|    24 diff=0
localidad         1177|  1177 diff=0
direccion         5975|  5927 diff=48
tipo_negocio        12|    12 diff=0
